# Synthetic QA Data Generation — Sharp CV-P09FX Manual

Uses a strong LLM via DeepInfra API to generate diverse question-answer pairs from the
cleaned Markdown manual. The goal is ~500 varied QA pairs so that key facts (22 inches /
559mm, 2 screws, drainage steps, etc.) appear from many angles — the repetition and
diversity that makes on-policy distillation actually land the facts.

**Strategy:** generate multiple passes over manual sections, each with a different
diversity instruction (direct lookup, metric/imperial, edge cases, procedural,
troubleshooting). This ensures the same spec gets covered from many question angles.

Output: `ac_manual_synth_tra.jsonl` and `ac_manual_synth_val.jsonl` in the same
ChatML format as the original dataset — drop-in replacement for the training notebook.


In [ ]:
# 1. Config
import re, json, random, time
import requests

# --- Fill these in ---
DEEPINFRA_API_KEY = ""
# ---------------------

# DeepSeek-V4-Flash: native reasoning model with 1M context on DeepInfra.
# reasoning_effort="high" tells the model to think carefully before answering.
GEN_MODEL        = "deepseek-ai/DeepSeek-V4-Flash"
REASONING_EFFORT = "high"

MANUAL_MD_PATH   = "sharp_cv_p09fx_manual_en.md"
OUTPUT_TRAIN     = "ac_manual_synth_tra.jsonl"
OUTPUT_VAL       = "ac_manual_synth_val.jsonl"

# Pairs per (diversity_mode, pass) call. Full manual fits in one request so
# we can ask for more pairs and get richer coverage per call.
N_PAIRS_PER_BATCH = 15
N_PASSES          = 3      # repeat all diversity modes this many times for volume
VAL_RATIO         = 0.15
SEED              = 42
API_RETRY_DELAY   = 5

SYSTEM = ("You are a helpful assistant for the Sharp CV-P09FX portable air conditioner. "
          "Answer questions accurately based on the product manual.")

random.seed(SEED)
print("Config OK.")


In [ ]:
# 2. Load manual
# 1M context means we pass the entire manual in every API call — no section splitting needed.
with open(MANUAL_MD_PATH, encoding="utf-8") as f:
    manual_text = f.read()

print(f"Manual loaded: {len(manual_text)} chars")
print(f"'559' in manual : {'559' in manual_text}")
print(f"'22 inches' in manual: {'22 inches' in manual_text.lower()}")


In [ ]:
# 3. DeepInfra API helper
def call_api(messages, max_tokens=2048, temperature=0.8, retries=3, timeout=120):
    """Call DeepInfra chat completions. Returns (reasoning, answer) tuple.
    reasoning_content is the native thinking trace; content is the final answer."""
    url = "https://api.deepinfra.com/v1/openai/chat/completions"
    headers = {
        "Authorization": f"Bearer {DEEPINFRA_API_KEY}",
        "Content-Type": "application/json",
    }
    payload = {
        "model": GEN_MODEL,
        "messages": messages,
        "max_tokens": max_tokens,
        "temperature": temperature,
        "reasoning_effort": REASONING_EFFORT,
    }
    for attempt in range(retries):
        try:
            r = requests.post(url, headers=headers, json=payload, timeout=timeout)
            r.raise_for_status()
            msg = r.json()["choices"][0]["message"]
            reasoning = msg.get("reasoning_content") or ""
            content   = msg.get("content") or ""
            return reasoning.strip(), content.strip()
        except requests.HTTPError as e:
            if r.status_code == 429 and attempt < retries - 1:
                print(f"Rate limited, waiting {API_RETRY_DELAY}s...")
                time.sleep(API_RETRY_DELAY)
            else:
                raise
        except requests.Timeout:
            if attempt < retries - 1:
                print(f"Timeout on attempt {attempt+1}, retrying...")
            else:
                print("All retries timed out.")
                return "", ""
    return "", ""

def format_answer(reasoning, content):
    """Combine reasoning trace + final answer into <think>...</think>answer format."""
    if reasoning:
        return f"<think>\n{reasoning}\n</think>\n{content}"
    return content


In [ ]:
# 4. Generation prompts + QA parser

DIVERSITY_MODES = [
    "Mix direct fact lookup questions ('What is...', 'How much...', 'What are...') "
    "with yes/no questions ('Does the unit require...', 'Is it possible to...').",

    "Focus on unit variations. Some questions use inches, some millimeters, some centimeters. "
    "Include questions that give a measurement and ask if it meets the requirement "
    "(e.g. 'My window is 550mm wide, can I install the panel?').",

    "Focus on edge cases and boundary conditions: what happens at the minimum or maximum "
    "value, what if a value is slightly too small or too large, what are the consequences "
    "of not meeting a requirement.",

    "Focus on procedural and step-by-step questions: how to perform an action, "
    "in what order to do steps, what tools or parts are needed.",

    "Focus on troubleshooting and practical scenarios: what could go wrong, "
    "what to check if something doesn't work, common mistakes to avoid.",
]

def build_messages(manual, diversity_instruction, n):
    # Manual goes in the system message — it is the same across all 15 calls so
    # DeepInfra's automatic prompt caching will cache it after the first call.
    # Only the user message (diversity instruction + task) changes each call.
    system_msg = (
        f"You are a precise technical assistant. "
        f"The following is the complete Sharp CV-P09FX portable air conditioner manual. "
        f"Use ONLY information from this manual when answering. Do not invent facts.\n\n"
        f"--- MANUAL START ---\n{manual}\n--- MANUAL END ---"
    )
    user_msg = (
        f"Generate exactly {n} diverse question-answer pairs about this air conditioner.\n\n"
        f"Diversity instruction: {diversity_instruction}\n\n"
        f"Rules:\n"
        f"- Questions and answers must be grounded in the manual above\n"
        f"- Keep answers concise (1-4 sentences)\n\n"
        f"Format:\n"
        f"Q: [question]\n"
        f"A: [answer]\n\n"
        f"Generate all {n} pairs now:"
    )
    return [
        {"role": "system", "content": system_msg},
        {"role": "user",   "content": user_msg},
    ]

def parse_qa_pairs(content):
    """Extract Q/A pairs from the model's content output."""
    pairs = []
    pattern = re.compile(r'Q:\s*(.+?)\nA:\s*(.+?)(?=\nQ:|\Z)', re.DOTALL)
    for m in pattern.finditer(content):
        q = m.group(1).strip().strip('"')
        a = m.group(2).strip().strip('"')
        if q and a and len(q) > 10 and len(a) > 10:
            pairs.append((q, a))
    return pairs


In [ ]:
# 5. Generate QA pairs
# N_PASSES x 5 modes x N_PAIRS_PER_BATCH = ~225 pairs per run.
# The manual is in the system message so DeepInfra caches it automatically after
# the first call — calls 2-15 only process the short user message, saving ~140k tokens.

all_pairs = []

total_batches = N_PASSES * len(DIVERSITY_MODES)
batch_num = 0

for pass_idx in range(N_PASSES):
    for mode_idx, diversity in enumerate(DIVERSITY_MODES):
        batch_num += 1
        print(f"[{batch_num}/{total_batches}] pass={pass_idx} mode={mode_idx} ... ", end="", flush=True)

        messages = build_messages(manual_text, diversity, N_PAIRS_PER_BATCH)

        try:
            reasoning, content = call_api(messages, max_tokens=4096, temperature=0.85)
            pairs_raw = parse_qa_pairs(content)
            for q, a in pairs_raw:
                all_pairs.append((q, format_answer(reasoning, a)))
            cached_note = "(cached)" if batch_num > 1 else "(first call)"
            print(f"got {len(pairs_raw)} pairs {cached_note} | reasoning: {len(reasoning)} chars (total: {len(all_pairs)})")
        except Exception as e:
            print(f"ERROR: {e}")

print(f"\nGeneration complete. Total raw pairs: {len(all_pairs)}")


In [ ]:
# 5b. Pass 2: per-question reasoning enrichment
# The batch reasoning traces from pass 1 are meta-reasoning ("let me plan 15 pairs...")
# not question-specific. Here we re-ask each question individually so DeepSeek's native
# reasoning_content is grounded in that specific question.
# Manual is still in the system message → cached for all ~200 calls.

def enrich_one(question):
    """Ask DeepSeek the question directly. Returns (reasoning, answer).
    reasoning_content will be question-specific, not batch meta-reasoning."""
    messages = [
        {"role": "system", "content": (
            f"You are a helpful assistant for the Sharp CV-P09FX portable air conditioner. "
            f"Answer questions accurately based ONLY on the product manual below.\n\n"
            f"--- MANUAL START ---\n{manual_text}\n--- MANUAL END ---"
        )},
        {"role": "user", "content": question},
    ]
    return call_api(messages, max_tokens=512, temperature=0.3, timeout=180)

enriched_pairs = []
failed = []
n = len(all_pairs)

for idx, (q, _) in enumerate(all_pairs):
    reasoning, content = enrich_one(q)
    if content:
        enriched_pairs.append((q, format_answer(reasoning, content)))
    else:
        # API failed — keep the original answer without a think block as fallback
        failed.append(idx)
        enriched_pairs.append((q, _))

    if (idx + 1) % 20 == 0 or idx == n - 1:
        print(f"  {idx+1}/{n} enriched | failed so far: {len(failed)}")

# Replace batch pairs with per-question enriched versions
all_pairs = enriched_pairs
print(f"\nPass 2 complete. {len(all_pairs)} pairs with question-specific reasoning.")
if failed:
    print(f"  {len(failed)} pairs kept original answer (no think block): indices {failed[:10]}{'...' if len(failed)>10 else ''}")


In [ ]:
# 6. Deduplicate + basic quality filter

def normalise(text):
    return re.sub(r'\s+', ' ', text.lower().strip())

def extract_final_answer(answer):
    """Return the text after </think>, or the full answer if no think block."""
    if '</think>' in answer:
        return answer.split('</think>', 1)[1].strip()
    return answer

seen_questions = set()
clean_pairs = []

for q, a in all_pairs:
    key = normalise(q)
    if key in seen_questions:
        continue
    final = extract_final_answer(a)
    if len(final.split()) < 4:
        continue
    if any(phrase in final.lower() for phrase in ["not mentioned", "not specified", "cannot determine"]):
        continue
    seen_questions.add(key)
    clean_pairs.append((q, a))

print(f"After dedup + filter: {len(clean_pairs)} pairs (removed {len(all_pairs) - len(clean_pairs)})")

# Coverage check on key specs
key_facts = ["22", "559", "screw", "drain", "filter", "exhaust", "install"]
for fact in key_facts:
    count = sum(1 for q, a in clean_pairs if fact.lower() in (q + a).lower())
    print(f"  '{fact}' appears in {count} pairs")


In [ ]:
# 7. Format to ChatML JSONL + train/val split + save

def to_chatml_jsonl(question, answer):
    """Format a QA pair in the same ChatML format as the original dataset."""
    text = (
        f"<|im_start|>system\n{SYSTEM}<|im_end|>\n"
        f"<|im_start|>user\n{question}<|im_end|>\n"
        f"<|im_start|>assistant\n{answer}<|im_end|>"
    )
    return json.dumps({"text": text}, ensure_ascii=False)

# Shuffle before splitting so val isn't biased toward the last sections.
pairs_shuffled = clean_pairs.copy()
random.shuffle(pairs_shuffled)

n_val   = max(10, int(len(pairs_shuffled) * VAL_RATIO))
n_train = len(pairs_shuffled) - n_val

train_pairs = pairs_shuffled[:n_train]
val_pairs   = pairs_shuffled[n_train:]

with open(OUTPUT_TRAIN, "w", encoding="utf-8") as f:
    for q, a in train_pairs:
        f.write(to_chatml_jsonl(q, a) + "\n")

with open(OUTPUT_VAL, "w", encoding="utf-8") as f:
    for q, a in val_pairs:
        f.write(to_chatml_jsonl(q, a) + "\n")

print(f"Saved {n_train} train pairs → {OUTPUT_TRAIN}")
print(f"Saved {n_val} val pairs   → {OUTPUT_VAL}")


In [ ]:
# 8. Preview a few examples to sanity-check quality
print("=== Sample train pairs ===\n")
for q, a in random.sample(train_pairs, min(5, len(train_pairs))):
    print(f"Q: {q}")
    print(f"A: {a}")
    print()

print("\n=== To use in the training notebook ===")
print(f"Upload {OUTPUT_TRAIN} and {OUTPUT_VAL} to Colab, then in the config cell set:")
print(f'  TRAIN_JSONL_URL = "{OUTPUT_TRAIN}"  # local path or upload to GitHub and use raw URL')
print(f'  VAL_JSONL_URL   = "{OUTPUT_VAL}"')
print(f"  EPOCHS = 3  # larger dataset needs fewer epochs")
